# all-reduce-eval-metrics — ex1: average a per-rank eval loss across ranks

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `all-reduce-eval-metrics`. Running the final beacon cell reports progress against the `Distributed: all_reduce eval metrics` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: all_reduce eval metrics` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`all-reduce-eval-metrics`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "all-reduce-eval-metrics"
DD_SUBTOPIC = "Distributed: all_reduce eval metrics"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## torch.distributed quick refresher

PyTorch's collective-communication library (`torch.distributed`, aliased `dist`) lets multiple processes coordinate over tensors. Each rank runs the same function in its own process; collectives operate in-place on tensors of identical shape across ranks.

**Backends.** `'nccl'` for multi-GPU (ARENA's setup), `'gloo'` for CPU (what these drills use — Colab CPU runtimes have no GPUs).

**Launch pattern.** Each test uses `mp.get_context('fork').Process` so worker fns defined in a notebook cell are picklable. Workers init the group, do their work, push results onto a `manager.Queue`, then destroy the group.

### This drill's atom: average eval metrics across ranks
During distributed evaluation, each rank computes a metric (loss, accuracy) on its own shard of the validation set. To get the global metric you must average across ranks:
```python
local_loss = compute_loss(local_batch)
loss_t = t.tensor([local_loss])
dist.all_reduce(loss_t, op=dist.ReduceOp.SUM)
loss_t /= world_size              # convert SUM → MEAN
global_loss = loss_t.item()
```
**Why wrap in a tensor.** `dist.all_reduce` requires a tensor input — raw Python floats can't be reduced directly. Build a singleton tensor, reduce it, unwrap with `.item()`.

**Why SUM-then-divide vs MEAN.** PyTorch's `ReduceOp` has no `MEAN` — the canonical idiom is sum then divide by `world_size`.

### Exercise 1 — average a per-rank eval loss across ranks

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `dist.all_reduce(SUM) / world_size` pattern to average a per-rank scalar eval loss into a global mean visible on every rank.
> Keywords: all_reduce, eval, metric, mean, wrap-scalar-tensor
> ```

**KCs targeted:** `wrap-float-in-tensor-for-reduce`, `sum-then-divide-by-world-size`

Implement `ex1_eval_metric_worker(rank, world_size, port, out_queue)`. Each rank:

1. Inits `gloo`.
2. Computes a fake per-rank loss: `local_loss = float(rank + 1)`. (Rank 0 → 1.0, rank 1 → 2.0, rank 2 → 3.0.)
3. **Averages across ranks via all_reduce + divide:**
   ```python
   loss_t = t.tensor([local_loss])
   dist.all_reduce(loss_t, op=dist.ReduceOp.SUM)
   loss_t /= world_size
   global_loss = loss_t.item()
   ```
4. Pushes `(rank, global_loss)` onto `out_queue`.
5. Destroys process group.

**Expected.** With `world_size=3`, mean of `[1, 2, 3]` = `2.0` on every rank. With `world_size=2`, mean of `[1, 2]` = `1.5`.

In [ ]:
def ex1_eval_metric_worker(rank, world_size, port, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    local_loss = float(rank + 1)
    loss_t = t.tensor([local_loss])
    dist.all_reduce(loss_t, op=dist.ReduceOp.SUM)
    loss_t /= world_size
    out_queue.put((rank, loss_t.item()))
    dist.destroy_process_group()


<details><summary>Solution</summary>

```python
def ex1_eval_metric_worker(rank, world_size, port, out_queue):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend='gloo', rank=rank, world_size=world_size,
                            timeout=datetime.timedelta(seconds=20))
    local_loss = float(rank + 1)
    loss_t = t.tensor([local_loss])
    dist.all_reduce(loss_t, op=dist.ReduceOp.SUM)
    loss_t /= world_size
    out_queue.put((rank, loss_t.item()))
    dist.destroy_process_group()
```

**Trap: forgetting the divide.** The most common bug. You'll see training logs where 'eval loss' silently scales with `world_size`. The fix is one line; the bug is invisible until someone notices the loss curve looks weirdly stable across scaling experiments.

**For batch-weighted metrics.** If ranks have unequal batch sizes (last batch problem), do a SUM all_reduce on numerator AND on count, then divide. Plain mean would over-weight ranks with smaller batches.

**Why on every rank not just rank-0.** `all_reduce` puts the result on EVERY rank (that's the 'all' part vs `reduce` to one). This is useful when downstream logic — early stopping, LR scheduling, checkpointing — needs the global metric on every rank.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()